In [15]:
import time
import multiprocessing
import pandas as pd
import numpy as np
from google.colab import drive


In [16]:
drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/PDC_CFILES/FraudShield_Banking_Data.csv'
data = pd.read_csv(file_path)

print(f"Dataset: {data.shape[0]:,} rows, {data.shape[1]} columns")
print(f"Columns: {list(data.columns)}\n")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset: 50,000 rows, 25 columns
Columns: ['Transaction_ID', 'Customer_ID', 'Transaction_Amount (in Million)', 'Transaction_Time', 'Transaction_Date', 'Transaction_Type', 'Merchant_ID', 'Merchant_Category', 'Transaction_Location', 'Customer_Home_Location', 'Distance_From_Home', 'Device_ID', 'IP_Address', 'Card_Type', 'Account_Balance (in Million)', 'Daily_Transaction_Count', 'Weekly_Transaction_Count', 'Avg_Transaction_Amount (in Million)', 'Max_Transaction_Last_24h (in Million)', 'Is_International_Transaction', 'Is_New_Merchant', 'Failed_Transaction_Count', 'Unusual_Time_Transaction', 'Previous_Fraud_Count', 'Fraud_Label']



In [17]:

def sequential_processing(data):
    # Operation 1: Filtering - transactions above 5000
    numeric_col = data.select_dtypes(include=[np.number]).columns[0]
    filtered = data[data[numeric_col] > 5000]

    # Operation 2: Aggregation - sum by category
    cat_col = data.select_dtypes(include=['object']).columns[0]
    sum_by_category = filtered.groupby(cat_col)[numeric_col].sum().to_dict()

    # Operation 3: Aggregation - average by second category
    cat_col2 = data.select_dtypes(include=['object']).columns[1] if len(data.select_dtypes(include=['object']).columns) > 1 else cat_col
    avg_by_category = filtered.groupby(cat_col2)[numeric_col].mean().to_dict()

    return {
        'filtered_count': len(filtered),
        'sum_by_category': sum_by_category,
        'avg_by_category': avg_by_category
    }


In [18]:
def chunk_data(data, num_chunks):
    chunk_size = len(data) // num_chunks
    chunks = []
    for i in range(num_chunks):
        start = i * chunk_size
        end = start + chunk_size if i < num_chunks - 1 else len(data)
        chunks.append(data.iloc[start:end].copy())
    return chunks


In [20]:
def process_chunk(chunk):
    numeric_col = chunk.select_dtypes(include=[np.number]).columns[0]
    filtered = chunk[chunk[numeric_col] > 5000]

    cat_col = chunk.select_dtypes(include=['object']).columns[0]
    sum_by_category = filtered.groupby(cat_col)[numeric_col].sum().to_dict()

    cat_col2 = chunk.select_dtypes(include=['object']).columns[1] if len(chunk.select_dtypes(include=['object']).columns) > 1 else cat_col
    avg_by_category = filtered.groupby(cat_col2)[numeric_col].mean().to_dict()

    return {
        'count': len(filtered),
        'sum_by_category': sum_by_category,
        'avg_by_category': avg_by_category
    }


In [21]:
def parallel_processing(data):
    num_processes = multiprocessing.cpu_count()
    chunks = chunk_data(data, num_processes)

    with multiprocessing.Pool(processes=num_processes) as pool:
        results = pool.map(process_chunk, chunks)

    total_count = sum(r['count'] for r in results)

    combined_sum = {}
    for r in results:
        for cat, val in r['sum_by_category'].items():
            combined_sum[cat] = combined_sum.get(cat, 0) + val

    combined_avg = {}
    count_per_cat = {}
    for r in results:
        for cat, val in r['avg_by_category'].items():
            if cat in combined_avg:
                combined_avg[cat] += val
                count_per_cat[cat] += 1
            else:
                combined_avg[cat] = val
                count_per_cat[cat] = 1

    for cat in combined_avg:
        combined_avg[cat] /= count_per_cat[cat]

    return {
        'filtered_count': total_count,
        'sum_by_category': combined_sum,
        'avg_by_category': combined_avg
    }

print("SEQUENTIAL PROCESSING")
start_time = time.time()
seq_results = sequential_processing(data)
seq_time = time.time() - start_time
print(f"Time: {seq_time:.4f} seconds")

print("\nPARALLEL PROCESSING")
start_time = time.time()
par_results = parallel_processing(data)
par_time = time.time() - start_time
print(f"Time: {par_time:.4f} seconds")

speedup = seq_time / par_time
num_cores = multiprocessing.cpu_count()
efficiency = speedup / num_cores

print("\n" + "="*50)
print("PERFORMANCE COMPARISON")
print("="*50)
print(f"{'Version':<20} {'Time (seconds)':<15}")
print("-"*50)
print(f"{'Sequential':<20} {seq_time:<15.4f}")
print(f"{'Parallel':<20} {par_time:<15.4f}")
print(f"{'Speedup':<20} {speedup:<15.2f}x")
print(f"{'Efficiency':<20} {efficiency:<15.2%}")
print("="*50)

print("\nRESULTS VERIFICATION")
print(f"Sequential filtered count: {seq_results['filtered_count']}")
print(f"Parallel filtered count: {par_results['filtered_count']}")
print(f"Results match: {seq_results['filtered_count'] == par_results['filtered_count']}")

SEQUENTIAL PROCESSING
Time: 0.2057 seconds

PARALLEL PROCESSING
Time: 0.8132 seconds

PERFORMANCE COMPARISON
Version              Time (seconds) 
--------------------------------------------------
Sequential           0.2057         
Parallel             0.8132         
Speedup              0.25           x
Efficiency           12.65%         

RESULTS VERIFICATION
Sequential filtered count: 49997
Parallel filtered count: 49997
Results match: True
